# Orientation Testing - Single Reflection Scenario

This notebook tests actor orientations for the Single Reflection scenario.
It logs all angles and saves a visualization without paths.

In [1]:
import os
import numpy as np
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera, mi
from heatmap_situations import situations
from sionna_utils import create_tx_actor, create_ris_actor, create_rx_actor

# Parameters
K = 4   # Number of antennas
N = 36  # Number of RIS elements

# Create output folder
os.makedirs("orientation_test", exist_ok=True)

print("✓ Setup complete")

✓ Setup complete


In [2]:
# Load Single Reflection scenario
situation = next((s for s in situations if s['simulation_name'] == 'Single Reflection'), None)

if not situation:
    print("❌ Single Reflection scenario not found!")
else:
    print(f"✓ Loaded scenario: {situation['simulation_name']}")
    print(f"  Size: {situation['width']}m × {situation['height']}m")
    print(f"  Buildings: {len(situation['buildings'])}")
    print(f"  RIS points: {len(situation['ris_points'])}")
    print(f"  Receivers: {len(situation['receivers'])}")

✓ Loaded scenario: Single Reflection
  Size: 20m × 20m
  Buildings: 2
  RIS points: 1
  Receivers: 2


In [3]:
# Extract scenario data
tx_point = situation['transmitter']
ris_points = situation['ris_points']
receivers = situation['receivers']
buildings = situation['buildings']

print("\n" + "="*80)
print("SCENARIO GEOMETRY")
print("="*80)

print(f"\nTransmitter (TX):")
print(f"  Position: ({tx_point['x']:.2f}, {tx_point['y']:.2f})")

print(f"\nRIS Points:")
for i, ris in enumerate(ris_points, 1):
    print(f"  P{i}: ({ris['x']:.2f}, {ris['y']:.2f})")

print(f"\nReceivers:")
for i, rx in enumerate(receivers, 1):
    print(f"  R{i}: ({rx['x']:.2f}, {rx['y']:.2f})")

print(f"\nBuildings:")
for i, building in enumerate(buildings, 1):
    print(f"  Building {i}: ({building['x']:.2f}, {building['y']:.2f}) "
          f"size {building['width']:.2f}×{building['height']:.2f} "
          f"rotation {building.get('rotation', 0):.1f}°")


SCENARIO GEOMETRY

Transmitter (TX):
  Position: (3.00, 3.00)

RIS Points:
  P1: (7.00, 9.00)

Receivers:
  R1: (16.00, 11.00)
  R2: (10.00, 18.00)

Buildings:
  Building 1: (0.00, 10.00) size 7.00×10.00 rotation 0.0°
  Building 2: (8.00, 0.00) size 12.00×8.00 rotation 0.0°


In [4]:
# Create actors and log their orientations
print("\n" + "="*80)
print("ACTOR ORIENTATIONS (CURRENT IMPLEMENTATION)")
print("="*80)

# Create TX actor
tx_actor = create_tx_actor(tx_point, ris_points, receivers, K)
print(f"\nTransmitter (TX):")
print(f"  Name: {tx_actor.name}")
print(f"  Position: {tx_actor.position}")
print(f"  Orientation (roll, yaw, pitch): {tx_actor.orientation}")
print(f"  Array size: {tx_actor.rows}×{tx_actor.cols} (linear)")
print(f"  → Yaw angle: {tx_actor.orientation[1]:.2f}°")

# Calculate expected direction
target = ris_points[0] if len(ris_points) > 0 else receivers[0]
vec = (target['x'] - tx_point['x'], target['y'] - tx_point['y'])
expected_angle = np.degrees(np.arctan2(vec[1], vec[0]))
print(f"  Expected direction to target: {expected_angle:.2f}° (toward P1)")
print(f"  Vector to target: ({vec[0]:.2f}, {vec[1]:.2f})")

# Create RIS actors
ris_actors = []
for i, ris_point in enumerate(ris_points, 1):
    ris_actor = create_ris_actor(f'P{i}', ris_point, tx_point, receivers, N)
    ris_actors.append(ris_actor)
    
    print(f"\nRIS {i} (P{i}):")
    print(f"  Name: {ris_actor.name}")
    print(f"  Position: {ris_actor.position}")
    print(f"  Orientation (roll, yaw, pitch): {ris_actor.orientation}")
    print(f"  Array size: {ris_actor.rows}×{ris_actor.cols} (planar)")
    print(f"  → Yaw angle: {ris_actor.orientation[1]:.2f}°")
    
    # Calculate expected directions
    vec_from_tx = (tx_point['x'] - ris_point['x'], tx_point['y'] - ris_point['y'])
    angle_from_tx = np.degrees(np.arctan2(vec_from_tx[1], vec_from_tx[0]))
    
    if len(receivers) > 0:
        vec_to_rx = (receivers[0]['x'] - ris_point['x'], receivers[0]['y'] - ris_point['y'])
        angle_to_rx = np.degrees(np.arctan2(vec_to_rx[1], vec_to_rx[0]))
        bisector = (angle_from_tx + angle_to_rx) / 2
        print(f"  Direction from TX: {angle_from_tx:.2f}°")
        print(f"  Direction to R1: {angle_to_rx:.2f}°")
        print(f"  Bisector angle: {bisector:.2f}° (expected normal ≈ bisector - 90°)")

# Create RX actors
rx_actors = []
for i, receiver in enumerate(receivers, 1):
    rx_actor = create_rx_actor(f'R{i}', receiver, ris_points, tx_point, K)
    rx_actors.append(rx_actor)
    
    print(f"\nReceiver {i} (R{i}):")
    print(f"  Name: {rx_actor.name}")
    print(f"  Position: {rx_actor.position}")
    print(f"  Orientation (roll, yaw, pitch): {rx_actor.orientation}")
    print(f"  Array size: {rx_actor.rows}×{rx_actor.cols} (linear)")
    print(f"  → Yaw angle: {rx_actor.orientation[1]:.2f}°")
    
    # Calculate expected direction
    target = ris_points[0] if len(ris_points) > 0 else tx_point
    vec = (target['x'] - receiver['x'], target['y'] - receiver['y'])
    expected_angle = np.degrees(np.arctan2(vec[1], vec[0]))
    print(f"  Expected direction to target: {expected_angle:.2f}° (toward P1)")
    print(f"  Vector to target: ({vec[0]:.2f}, {vec[1]:.2f})")


ACTOR ORIENTATIONS (CURRENT IMPLEMENTATION)

Transmitter (TX):
  Name: T
  Position: (3, 3, 1.5)
  Orientation (roll, yaw, pitch): (0.0, 56.309932474020215, 0.0)
  Array size: 1×4 (linear)
  → Yaw angle: 56.31°
  Expected direction to target: 56.31° (toward P1)
  Vector to target: (4.00, 6.00)

RIS 1 (P1):
  Name: P1
  Position: (7, 9, 1.5)
  Orientation (roll, yaw, pitch): (0.0, -40.82156904143252, 0.0)
  Array size: 6×6 (planar)
  → Yaw angle: -40.82°
  Direction from TX: -123.69°
  Direction to R1: 12.53°
  Bisector angle: -55.58° (expected normal ≈ bisector - 90°)

Receiver 1 (R1):
  Name: R1
  Position: (16, 11, 1.5)
  Orientation (roll, yaw, pitch): (0.0, -167.47119229084848, 0.0)
  Array size: 1×4 (linear)
  → Yaw angle: -167.47°
  Expected direction to target: -167.47° (toward P1)
  Vector to target: (-9.00, -2.00)

Receiver 2 (R2):
  Name: R2
  Position: (10, 18, 1.5)
  Orientation (roll, yaw, pitch): (0.0, -108.43494882292202, 0.0)
  Array size: 1×4 (linear)
  → Yaw angle: -

In [5]:
# Load and configure scene
print("\n" + "="*80)
print("RENDERING SCENE")
print("="*80)

scene_path = f"mesh_scene/{situation['simulation_name']}.xml"
scene = load_scene(scene_path)
scene.frequency = 3.5e9

# Configure antenna arrays
scene.tx_array = PlanarArray(
    num_rows=1,
    num_cols=K,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization="V"
)

scene.rx_array = PlanarArray(
    num_rows=1,
    num_cols=K,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="iso",
    polarization="V"
)

# Add TX
tx = Transmitter(
    name=tx_actor.name,
    position=mi.Point3f(list(tx_actor.position)),
    orientation=mi.Point3f(list(tx_actor.orientation)),
    display_radius=0.5,
    color=[1.0, 0.0, 0.0]  # Red
)
scene.add(tx)

# Add RIS points
for ris_actor in ris_actors:
    ris = Transmitter(
        name=ris_actor.name,
        position=mi.Point3f(list(ris_actor.position)),
        orientation=mi.Point3f(list(ris_actor.orientation)),
        display_radius=0.4,
        color=[0.0, 0.0, 1.0]  # Blue
    )
    scene.add(ris)

# Add receivers
for rx_actor in rx_actors:
    rx = Receiver(
        name=rx_actor.name,
        position=mi.Point3f(list(rx_actor.position)),
        orientation=mi.Point3f(list(rx_actor.orientation)),
        display_radius=0.3,
        color=[0.0, 1.0, 0.0]  # Green
    )
    scene.add(rx)

# Setup camera
center_x = situation['width'] / 2
center_y = situation['height'] / 2
max_dim = max(situation['width'], situation['height'])
camera_height = max_dim * 2.5
camera_offset = max_dim * 0.15

camera = Camera(
    position=mi.Point3f([center_x - camera_offset, center_y - camera_offset, camera_height]),
    look_at=mi.Point3f([center_x, center_y, 5.0])
)

# Render WITHOUT paths
output_file = "orientation_test/single_reflection_orientation_test.png"
print(f"\nRendering scene (no paths)...")
print(f"Output: {output_file}")

scene.render_to_file(
    camera=camera,
    filename=output_file,
    resolution=(1400, 1000),
    num_samples=512
)

print(f"\n✅ Rendering complete!")
print(f"✅ Image saved to: {output_file}")
print(f"\nThe lines coming from each colored sphere show the CURRENT orientation (yaw angle).")
print(f"Compare the angles logged above with the visual representation to debug.")


RENDERING SCENE

Rendering scene (no paths)...
Output: orientation_test/single_reflection_orientation_test.png

✅ Rendering complete!
✅ Image saved to: orientation_test/single_reflection_orientation_test.png

The lines coming from each colored sphere show the CURRENT orientation (yaw angle).
Compare the angles logged above with the visual representation to debug.


## Summary

This notebook:
1. Loaded the "Single Reflection" scenario
2. Created all actors with the current orientation logic
3. Logged all positions and orientation angles
4. Rendered the scene showing actor positions and orientations (the lines)
5. Saved the image to `orientation_test/single_reflection_orientation_test.png`

**What to check:**
- Compare the yaw angles printed above with the line directions in the image
- TX (red) should point toward P1 (blue)
- P1 (blue) should be oriented to reflect from TX to R1 (bisector - 90°)
- R1 (green) should point toward P1

**If the lines don't match the expected directions:**
- Try adding/subtracting 90° in `sionna_utils.py:calculate_orientation()`
- Or reverse the vector direction in the calculation